# Unsupervised — Pola Sekuensial ("setelah A, orang biasanya B")
**Kapan dipakai:** data berisi urutan kejadian per orang (film yang ditonton, halaman yang
dibuka, produk yang dibeli dari waktu ke waktu) dan diminta pola urutan.

**Idenya:** beda dengan aturan asosiasi biasa, di sini **urutan waktu** dipakai. Pola A → B
dihitung bila B muncul paling jauh `WINDOW` langkah sesudah A pada urutan satu orang.
- **support** = porsi orang yang punya pola itu.
- **confidence** = dari orang yang pernah A, berapa porsi yang lalu B.
- **lift** = confidence dibagi porsi B biasa. Lift > 1 = B lebih sering muncul sesudah A.

**Langkah:**
1. Urutkan kejadian per orang berdasarkan waktu.
2. Hitung pola di 70% orang, cek ulang confidence-nya di 30% sisanya.
3. Hitung ulang dengan semua orang, simpan pola.

**Data demo:** MovieLens 100k (urut `timestamp` per user). Banyak rating diberi pada detik
yang sama, jadi urutan dalam satu sesi tidak pasti.

In [1]:
import numpy as np
import pandas as pd

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/recsys/ml-100k"   # GANTI saat lomba
EVENTS_CSV = f"{DATA_FOLDER}/ratings.csv"
ITEMS_CSV = f"{DATA_FOLDER}/movies.csv"

USER_COL = "user"
ITEM_COL = "item"
TIME_COL = "timestamp"
TITLE_COL = "title"
WINDOW = 10          # B dihitung bila muncul paling jauh 10 langkah sesudah A
MIN_USERS = 20       # pola harus dimiliki minimal 20 orang
SEED = 42

## Baca data

In [3]:
events = pd.read_csv(EVENTS_CSV)
items = pd.read_csv(ITEMS_CSV)
titles = pd.Series(items[TITLE_COL].to_numpy(), index=items[ITEM_COL])
events = events.sort_values([USER_COL, TIME_COL, ITEM_COL], kind="stable")
print(events.shape)
print("user:", events[USER_COL].nunique(), "| item:", events[ITEM_COL].nunique())
events.head()

(100000, 4)
user: 943 | item: 1682


,user,item,rating,timestamp
59972,1,168,5,874965478
92487,1,172,5,874965478
74577,1,165,5,874965518
48214,1,156,4,874965556
22971,1,166,5,874965677


## Pilih data urutan (satu baris aktif)

In [4]:
sequence = events
# sequence = events[events["rating"] >= 4]       # hanya film yang disukai

## Langkah 1 — hitung pola A → B
Tiap orang dihitung sekali per pola, walau polanya muncul berkali-kali di urutannya.

In [5]:
def pattern_table(table):
    pair_users = {}
    item_users = {}
    for _, user_items in table.groupby(USER_COL)[ITEM_COL]:
        user_items = user_items.tolist()
        user_pairs = set()
        for i in range(len(user_items)):
            for later in user_items[i + 1:i + 1 + WINDOW]:
                user_pairs.add((user_items[i], later))
        for pair in user_pairs:
            pair_users[pair] = pair_users.get(pair, 0) + 1
        for item in set(user_items):
            item_users[item] = item_users.get(item, 0) + 1
    patterns = pd.DataFrame(list(pair_users.keys()), columns=["A", "B"])
    patterns["n_users"] = list(pair_users.values())
    patterns["support"] = patterns["n_users"] / table[USER_COL].nunique()
    patterns["confidence"] = patterns["n_users"] / patterns["A"].map(item_users)
    b_share = patterns["B"].map(item_users) / table[USER_COL].nunique()
    patterns["lift"] = patterns["confidence"] / b_share
    return patterns


# coba pada 3 orang pertama
first_users = sequence[USER_COL].unique()[:3]
print(pattern_table(sequence[sequence[USER_COL].isin(first_users)]).head())

     A    B  n_users   support  confidence  lift
0  166  181        1  0.333333         1.0   1.5
1  240  130        1  0.333333         1.0   3.0
2  226  232        1  0.333333         1.0   3.0
3  186  159        1  0.333333         1.0   3.0
4  252  104        1  0.333333         1.0   3.0


## Langkah 2 — cari pola di 70% orang, cek di 30% sisanya
Pola yang bagus tetap punya confidence tinggi di orang lain (bukan kebetulan).

In [6]:
rng = np.random.default_rng(SEED)
all_users = sequence[USER_COL].unique()
train_users = rng.choice(all_users, int(len(all_users) * 0.7), replace=False)
is_train = sequence[USER_COL].isin(train_users)

patterns = pattern_table(sequence[is_train])
patterns = patterns[(patterns["n_users"] >= MIN_USERS) & (patterns["lift"] > 1)]
patterns = patterns.sort_values("confidence", ascending=False)
print("jumlah pola:", len(patterns))

test_patterns = pattern_table(sequence[~is_train])
test_confidence = {}
for item_a, item_b, value in zip(test_patterns["A"], test_patterns["B"],
                                 test_patterns["confidence"]):
    test_confidence[(item_a, item_b)] = value

top = patterns.head(20).copy()
top_test = []
for item_a, item_b in zip(top["A"], top["B"]):
    top_test.append(test_confidence.get((item_a, item_b), 0))
top["confidence_test"] = top_test
top["title_A"] = top["A"].map(titles)
top["title_B"] = top["B"].map(titles)
print("rata confidence train:", round(top["confidence"].mean(), 3))
print("rata confidence uji  :", round(top["confidence_test"].mean(), 3))
top[["title_A", "title_B", "n_users", "confidence", "confidence_test", "lift"]].round(3)

jumlah pola: 55


rata confidence train: 0.275
rata confidence uji  : 0.208


,title_A,title_B,n_users,confidence,confidence_test,lift
30665,Good Will Hunting (1997),Titanic (1997),60,0.435,0.333,1.176
416,Murder at 1600 (1997),Dante's Peak (1997),58,0.369,0.311,1.401
19356,Dante's Peak (1997),Volcano (1997),57,0.328,0.409,1.441
29043,Event Horizon (1997),Volcano (1997),24,0.300,0.085,1.320
17218,Murder at 1600 (1997),Volcano (1997),47,0.299,0.262,1.317
12101,Star Trek: The Motion Picture (1979),Star Trek V: The Final Frontier (1989),22,0.282,0.205,4.231
271,"Devil's Own, The (1997)",Dante's Peak (1997),47,0.278,0.169,1.055
16040,Star Trek VI: The Undiscovered Country (1991),Star Trek III: The Search for Spock (1984),29,0.276,0.232,1.599
35504,Bram Stoker's Dracula (1992),Interview with the Vampire (1994),20,0.274,0.128,2.055
311,"Devil's Own, The (1997)",Murder at 1600 (1997),44,0.260,0.155,1.094


## Langkah 3 — hitung ulang dengan semua orang, simpan

In [7]:
patterns = pattern_table(sequence)
patterns = patterns[(patterns["n_users"] >= MIN_USERS) & (patterns["lift"] > 1)]
patterns = patterns.sort_values("confidence", ascending=False)

submission = pd.DataFrame()
submission["A"] = patterns["A"].map(titles)
submission["B"] = patterns["B"].map(titles)
submission["support"] = patterns["support"]
submission["confidence"] = patterns["confidence"]
submission["lift"] = patterns["lift"]
submission.to_csv("submission.csv", index=False)
submission.head(10)

,A,B,support,confidence,lift
36919,Good Will Hunting (1997),Titanic (1997),0.084836,0.404040,1.088600
3638,Murder at 1600 (1997),Dante's Peak (1997),0.081654,0.353211,1.387825
23369,Dante's Peak (1997),Volcano (1997),0.089077,0.350000,1.507078
28127,In the Company of Men (1997),Cop Land (1997),0.021209,0.303030,1.632900
21291,Murder at 1600 (1997),Volcano (1997),0.066808,0.288991,1.244376
19670,Good Will Hunting (1997),Apt Pupil (1998),0.056204,0.267677,1.577620
20126,Star Trek VI: The Undiscovered Country (1991),Star Trek III: The Search for Spock (1984),0.044539,0.260870,1.438596
16229,Star Trek: The Motion Picture (1979),Star Trek V: The Final Frontier (1989),0.031813,0.256410,3.838014
16835,Mother (1996),Murder at 1600 (1997),0.045599,0.254438,1.100619
21453,"Omen, The (1976)",Candyman (1992),0.022269,0.247059,3.584253
